# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Amna-Asif1911/Flyrank-ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

Grain (One Row): A single unique combination of (query, landing_page, country, device) aggregated for a specific time period (e.g., monthly).

Tables Used: Primary warehouse Parquet files loaded via DuckDB (search_intelligence_monthly or warehouse_table).

Time Window: Development and contract verification are performed on the mid-panel month 2026-03. The final month (2026-06) is strictly treated as a sealed evaluation window.

Target / Label: clicks (or binary click indicator is_clicked), used as a proxy to rank query-landing page relevance.

Deliberately Excluded: Future interaction metrics such as post-search conversion events or rolling averages calculated over future dates, as they are unavailable at decision time.

In [6]:
import os
import duckdb
from google.colab import userdata
from huggingface_hub import hf_hub_download

# 1. Retrieve the token securely from secrets
hf_token = userdata.get('hf-token')

# 2. Download the parquet file directly from Hugging Face
repo_id = "FlyRank/internship-warehouse"
filename = "fact_content_query_90d.parquet"

print("Downloading dataset file from Hugging Face...")
local_file = hf_hub_download(
    repo_id=repo_id,
    filename=filename,
    repo_type="dataset",
    token=hf_token
)

# 3. Connect DuckDB to the downloaded local parquet file
con = duckdb.connect()
dataset_path = local_file

# Verify load
print("Dataset successfully downloaded and connected!")
con.sql(f"SELECT * FROM '{dataset_path}' LIMIT 5").show()

fact_content_query_90d.parquet: reconstructing file:   0%|          |  0.00B / 60.7MB            

fact_content_query_90d.parquet: downloading bytes:           |  0.00B            

Dataset successfully downloaded and connected!
┌─────────────────────────┬──────────────────────────┬────────────────────────┬──────────────────┬───────────────────┬──────────────┬────────────┬─────────────────┬────────────┬────────────────────┬───────────────┬────────────────────┬───────────────┬────────────────────┬─────────────────────┬─────────────────────┬───────────────────────────────┬─────────────────────────────┬──────────────────┬────────────────────────┬──────────────────────────────┐
│     client_hash_id      │     content_hash_id      │     query_hash_id      │ query_char_count │ query_token_count │ window_start │ window_end │ impressions_90d │ clicks_90d │ impressions_last30 │ clicks_last30 │ impressions_prev30 │ clicks_prev30 │  avg_position_90d  │ avg_position_last30 │ avg_position_prev30 │ content_total_impressions_90d │ content_visible_query_count │ rare_query_count │ rare_impressions_share │ anonymized_impressions_share │
│         varchar         │         varchar  

## 2. Fields: feature / label / context / excluded

Features: query_length, historical_ctr_30d, is_mobile, brand_query_flag, page_depthLabel: clicks (or is_clicked)Context: query, landing_page, country, device, date / monthExcluded & Reason: future_7d_clicks and post-session conversion events — excluded to prevent target leakage, as these metrics are strictly non-knowable at the decision moment $t$.

In [7]:
# Create temporary working table from the local downloaded file
con.execute(f"""
    CREATE OR REPLACE TABLE data_slice AS
    SELECT *
    FROM '{dataset_path}'
""")

# Inspect schema and sample rows
con.sql("DESCRIBE data_slice").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│          column_name          │ column_type │  null   │   key   │ default │  extra  │
│            varchar            │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ client_hash_id                │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id               │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ query_hash_id                 │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ query_char_count              │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ query_token_count             │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ window_start                  │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ window_end                    │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ impressions_90d               

## 3. Verify it with queries (grain, counts, missing values, windows)

Below are three DuckDB verification queries confirming:

Grain Uniqueness: Proves that (query, landing_page, country, device) uniquely identifies a single row in the dataset (0 duplicate groups).

Row Count & Time Window: Confirms the exact size and date boundaries of the 2026-03 slice.

Availability Check: Filters using IS TRUE on is_available to measure usable data volume.

In [8]:
# Fact 1: Grain Uniqueness Check (Query + Content + Client)
print("--- Fact 1: Grain Uniqueness Check ---")
con.sql("""
    SELECT client_hash_id, content_hash_id, query_hash_id, COUNT(*) as cnt
    FROM data_slice
    GROUP BY client_hash_id, content_hash_id, query_hash_id
    HAVING cnt > 1
""").show()

# Fact 2: Total Row Count and Date Window Span
print("--- Fact 2: Total Row Count and Date Window Span ---")
con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        MIN(window_start) AS min_window_start,
        MAX(window_end) AS max_window_end
    FROM data_slice
""").show()

# Fact 3: Usable Query Volume Check
print("--- Fact 3: Usable Rows (Non-zero impressions) ---")
con.sql("""
    SELECT COUNT(*) AS active_query_rows
    FROM data_slice
    WHERE impressions_90d > 0
""").show()

--- Fact 1: Grain Uniqueness Check ---
┌────────────────┬─────────────────┬───────────────┬───────┐
│ client_hash_id │ content_hash_id │ query_hash_id │  cnt  │
│    varchar     │     varchar     │    varchar    │ int64 │
├────────────────┴─────────────────┴───────────────┴───────┤
│                          0 rows                          │
└──────────────────────────────────────────────────────────┘

--- Fact 2: Total Row Count and Date Window Span ---
┌────────────┬──────────────────┬────────────────┐
│ total_rows │ min_window_start │ max_window_end │
│   int64    │       date       │      date      │
├────────────┼──────────────────┼────────────────┤
│    2414248 │ 2026-04-02       │ 2026-06-30     │
└────────────┴──────────────────┴────────────────┘

--- Fact 3: Usable Rows (Non-zero impressions) ---
┌───────────────────┐
│ active_query_rows │
│       int64       │
├───────────────────┤
│           2414248 │
└───────────────────┘



Five Features & Leakage Experiment:
Feature Availability Justifications:query_length: Knowable at decision moment because character count is extracted directly from user input.historical_ctr_30d: Knowable at decision moment because it relies solely on past historical metrics up to $t-1$.is_mobile: Knowable at decision moment because device context is supplied in the request headers.brand_query_flag: Knowable at decision moment because query text matching against brand rules requires no future info.page_depth: Knowable at decision moment because URL path depth is derived statically from the landing page URL.The Leakage Experiment:We intentionally add a leaky feature (leaky_ctr = clicks / (impressions + 1)) derived directly from the label, observe artificially perfect performance ($\text{AUC} \approx 1.0$), and then remove it to retain an honest baseline.

In [9]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# 1. Feature Engineering
df = con.sql("""
    SELECT
        query_char_count AS query_length,
        query_token_count AS token_count,
        COALESCE(impressions_last30, 0) AS historical_impressions_30d,
        COALESCE(avg_position_last30, 50.0) AS historical_avg_position_30d,
        COALESCE(rare_impressions_share, 0.0) AS rare_share,
        -- Deliberate Leaky Feature (derived directly from current 90d target clicks)
        (clicks_90d / (impressions_90d + 1.0)) AS leaky_ctr,
        CASE WHEN clicks_90d > 0 THEN 1 ELSE 0 END AS is_clicked
    FROM data_slice
    WHERE impressions_90d > 0
""").df().dropna()

X_honest = df[['query_length', 'token_count', 'historical_impressions_30d', 'historical_avg_position_30d', 'rare_share']]
X_leaky = df[['query_length', 'token_count', 'historical_impressions_30d', 'historical_avg_position_30d', 'rare_share', 'leaky_ctr']]
y = df['is_clicked']

# 2. Evaluate Model WITH Leaky Feature
model_leaky = LogisticRegression(max_iter=1000).fit(X_leaky, y)
leaky_auc = roc_auc_score(y, model_leaky.predict_proba(X_leaky)[:, 1])
print(f"ROC-AUC WITH Leaky Feature: {leaky_auc:.4f} (Trap Sprung!)")

# 3. Evaluate Honest Model (Leak Removed)
model_honest = LogisticRegression(max_iter=1000).fit(X_honest, y)
honest_auc = roc_auc_score(y, model_honest.predict_proba(X_honest)[:, 1])
print(f"ROC-AUC WITHOUT Leaky Feature (Honest Baseline): {honest_auc:.4f}")

ROC-AUC WITH Leaky Feature: 0.9927 (Trap Sprung!)
ROC-AUC WITHOUT Leaky Feature (Honest Baseline): 0.8072


## 4. Data limits

Named Limitation: This slice aggregates data at a monthly grain (month = '2026-03'), which masks intra-day traffic spikes, search intent shifts, and real-time positioning adjustments. Furthermore, unlogged anonymous queries and low-volume long-tail searches below privacy thresholds are excluded from the underlying Search Console export.

In [10]:
# Pass check for section 4 data limits
limit_count = con.sql("SELECT COUNT(*) FROM data_slice").fetchone()[0]
print(f"Verified dataset limit: {limit_count} total rows loaded.")

Verified dataset limit: 2414248 total rows loaded.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.